<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/KD_apple_Resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip Apple.zip

Streaming output truncated to the last 5000 lines.
  inflating: Apple/Train/Cedar Apple Rust/0cd24b0c-0a9d-483f-8734-5c08988e029f___FREC_C.Rust 3762_90deg.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0cd24b0c-0a9d-483f-8734-5c08988e029f___FREC_C.Rust 3762_new30degFlipTB.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0cd24b0c-0a9d-483f-8734-5c08988e029f___FREC_C.Rust 3762_newGRR.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0ce943e7-3fed-41cb-8430-0e0f54ff2bc4___FREC_C.Rust 0014.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0ce943e7-3fed-41cb-8430-0e0f54ff2bc4___FREC_C.Rust 0014_90deg.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0ce943e7-3fed-41cb-8430-0e0f54ff2bc4___FREC_C.Rust 0014_new30degFlipLR.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0ce943e7-3fed-41cb-8430-0e0f54ff2bc4___FREC_C.Rust 0014_new30degFlipTB.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0ce943e7-3fed-41cb-8430-0e0f54ff2bc4___FREC_C.Rust 0014_newGRR.JPG  
  inflating: Apple/Train/Cedar Apple Rust/0ce9

In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
!mv PlantDoc-Dataset /content/PlantDoc
!ls /content/PlantDoc

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 32.42 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc-Dataset  PlantDoc_Examples.png  README.md  test  train


In [ ]:
import os
import tensorflow as tf
import numpy as np
import tempfile

from tensorflow.keras.models import load_model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, accuracy_score

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [ ]:
teacher_model_path = "/content/model.h5"
apple_data_dir = "/content/Apple"
plantdoc_train = "/content/PlantDoc/train"

In [ ]:
teacher_model = load_model(teacher_model_path)
teacher_model.trainable = False
print("Teacher model loaded.")

Teacher model loaded.


In [ ]:
img_size = (224, 224)
batch_size = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(apple_data_dir, "Train"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(apple_data_dir, "Test"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"
)

class_names_apple = train_ds.class_names
print("Apple dataset classes:", class_names_apple)

Found 5784 files belonging to 3 classes.
Found 146 files belonging to 3 classes.
Apple dataset classes: ['Apple Scab', 'Cedar Apple Rust', 'Healthy']


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

def prepare(ds, training=False):
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.map(lambda x, y: (preprocess_input(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare(train_ds, training=True)
test_ds = prepare(test_ds, training=False)

In [ ]:
student_base = ResNet50(
    include_top=False,
    input_shape=(224, 224, 3),
    weights="imagenet"
)

student_base.trainable = True   # You can freeze initially if needed

student_model = tf.keras.Sequential([
    student_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation="softmax")
])

print("ResNet50 student model created.")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
ResNet50 student model created.


In [ ]:
def normalize_name(n):
    return n.lower().replace("_", " ").replace("-", " ").strip()

class_names_plantdoc = [
    c for c in os.listdir(plantdoc_train)
    if os.path.isdir(os.path.join(plantdoc_train, c))
]

class_names_plantdoc_norm = [normalize_name(c) for c in class_names_plantdoc]

In [ ]:
mapping = {
    "Apple Scab": "Apple Scab Leaf",
    "Cedar Apple Rust": "Apple rust leaf",
    "Healthy": "Apple leaf"
}

relevant_indices = [
    class_names_plantdoc_norm.index(normalize_name(mapping[c]))
    for c in class_names_apple
]

print("Relevant teacher class indices:", relevant_indices)

Relevant teacher class indices: [22, 18, 20]


In [ ]:
T = 5.0
alpha = 0.7

optimizer = tf.keras.optimizers.Adam(1e-4)
ce_loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

In [ ]:
@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:

        # Teacher forward pass
        teacher_logits = teacher_model(images, training=False)
        teacher_probs = tf.nn.softmax(teacher_logits / T)
        teacher_probs_relevant = tf.gather(teacher_probs, relevant_indices, axis=1)

        # Student forward pass
        student_logits = student_model(images, training=True)
        student_probs = tf.nn.softmax(student_logits / T)

        # Normalize
        teacher_probs_relevant /= tf.reduce_sum(teacher_probs_relevant, axis=1, keepdims=True)
        student_probs /= tf.reduce_sum(student_probs, axis=1, keepdims=True)

        eps = 1e-7
        teacher_probs_relevant = tf.clip_by_value(teacher_probs_relevant, eps, 1.0)
        student_probs = tf.clip_by_value(student_probs, eps, 1.0)

        # Symmetric KL divergence
        kl_forward = tf.reduce_sum(
            teacher_probs_relevant * tf.math.log(teacher_probs_relevant / student_probs), axis=1
        )
        kl_backward = tf.reduce_sum(
            student_probs * tf.math.log(student_probs / teacher_probs_relevant), axis=1
        )

        kl_loss = tf.reduce_mean(0.5 * (kl_forward + kl_backward))
        soft_loss = kl_loss * (T ** 2)
        hard_loss = ce_loss_fn(labels, student_logits)

        loss = alpha * soft_loss + (1 - alpha) * hard_loss

    grads = tape.gradient(loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, student_model.trainable_variables))

    return tf.reduce_mean(loss)

In [ ]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0.0
    steps = 0

    for step, (images, labels) in enumerate(train_ds):
        loss = train_step(images, labels)
        total_loss += float(loss)
        steps += 1

        if step % 100 == 0:
            tf.print(f"Epoch {epoch+1}, Step {step}, Loss:", loss)

    avg_loss = total_loss / steps
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

Epoch 1, Step 0, Loss: 0.611403227
Epoch 1, Step 100, Loss: 0.172009841
Epoch 1, Step 200, Loss: 0.202237904
Epoch 1, Step 300, Loss: 0.158496737
Epoch 1, Step 400, Loss: 0.169739515
Epoch 1, Step 500, Loss: 0.19269903
Epoch 1, Step 600, Loss: 0.155771062
Epoch 1, Step 700, Loss: 0.156904086
Epoch 1/5 - Avg Loss: 0.1742
Epoch 2, Step 0, Loss: 0.147172809
Epoch 2, Step 100, Loss: 0.148827195
Epoch 2, Step 200, Loss: 0.148782641
Epoch 2, Step 300, Loss: 0.160652235
Epoch 2, Step 400, Loss: 0.149190709
Epoch 2, Step 500, Loss: 0.157537043
Epoch 2, Step 600, Loss: 0.14815709
Epoch 2, Step 700, Loss: 0.16275242
Epoch 2/5 - Avg Loss: 0.1513
Epoch 3, Step 0, Loss: 0.148948282
Epoch 3, Step 100, Loss: 0.145311803
Epoch 3, Step 200, Loss: 0.146779105
Epoch 3, Step 300, Loss: 0.147889078
Epoch 3, Step 400, Loss: 0.146022603
Epoch 3, Step 500, Loss: 0.150274232
Epoch 3, Step 600, Loss: 0.148371518
Epoch 3, Step 700, Loss: 0.147728
Epoch 3/5 - Avg Loss: 0.1484
Epoch 4, Step 0, Loss: 0.146805778
Ep

In [ ]:
y_true, y_pred = [], []

for images, labels in test_ds:
    preds = student_model.predict(images, verbose=0)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

print("\nFINAL EVALUATION ON APPLE:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names_apple))


FINAL EVALUATION ON APPLE:
Accuracy: 1.0
                  precision    recall  f1-score   support

      Apple Scab       1.00      1.00      1.00        51
Cedar Apple Rust       1.00      1.00      1.00        44
         Healthy       1.00      1.00      1.00        51

        accuracy                           1.00       146
       macro avg       1.00      1.00      1.00       146
    weighted avg       1.00      1.00      1.00       146



In [ ]:
with tempfile.NamedTemporaryFile(delete=False, suffix=".h5") as tmp:
    student_model.save(tmp.name)
    model_size = os.path.getsize(tmp.name) / (1024 * 1024)

print(f"\nStudent model size: {model_size:.2f} MB")


Student model size: 90.38 MB


In [ ]:
student_model.save("/content/student_model_apple_resnet50.h5")
print("Student model saved as student_model_apple_resnet50.h5")

from google.colab import files
files.download("/content/student_model_apple_resnet50.h5")

Student model saved as student_model_apple_resnet50.h5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>